In [43]:
# This notebook is a work in progress
_cf_value_cache = {}

def cf_value(terms):
    # return continued_fraction of terms as a real number (cached)
    key = tuple(tuple(part) for part in terms)
    if key not in _cf_value_cache:
        _cf_value_cache[key] = continued_fraction(terms).n()
    return _cf_value_cache[key]

def cf_extrema(term_lists):
    # return (min_value, min_terms), (max_value, max_terms) for a list of CF term lists
    valued = [(cf_value(terms), terms) for terms in term_lists]
    return (
        min(valued, key=lambda item: item[0]),
        max(valued, key=lambda item: item[0]),
    )

def sr_extrema(sr_candidates, component):
    # component: 0 for s, 1 for r; sr_candidates is e.g. sr_1[j] or sr_2[j]
    return cf_extrema(pair[component] for pair in sr_candidates)

def check_ii(h, k, sr_begin, sr_end):
    # verify connectivity condition (ii) between sr_1[0] and sr_2[-1]
    (s_lo, _), (s_hi, _) = sr_extrema(sr_begin, 0)
    (r_lo, _), (r_hi, _) = sr_extrema(sr_begin, 1)
    (s2_lo, _), (s2_hi, _) = sr_extrema(sr_end, 0)
    (r2_lo, _), (r2_hi, _) = sr_extrema(sr_end, 1)

    s_ok = (s_hi <= s2_lo) if h % 2 == 0 else (s_lo >= s2_hi)
    r_ok = (r_hi <= r2_lo) if k % 2 == 0 else (r_lo >= r2_hi)
    return {"s": s_ok, "r": r_ok}

def G(r, s, t, v, xv=None):
    # xv=None returns symbolic expression in x; otherwise numeric value
    var('x')
    rr, ss, tt, vv = var('rr ss tt vv')
    Gx = ((1 + rr*x)*(1 + ss*x)/((1 + tt*x)*(1 + vv*x))).subs(rr=r, ss=s, tt=t, vv=v)
    return Gx if xv is None else RR(Gx(x=xv))

def H(s_1, s_2, r_2, r_1, c_plus, c_minus):
    eta_1, eta_2 = cf_value([(0,), (1, 3)]), cf_value([(0,), (3, 1)])
    return (
        ((s_1 - s_2) / (r_2 - r_1))
        * G(r_2, r_1, eta_1, eta_2, c_plus)
        * G(eta_1, eta_2, s_1, s_2, c_minus)
    )

def _g_extrema_on_interval(r, s, t, v, x_range):
    lo, hi = x_range
    if r == t and s == v:
        # G is identically 1; skip solve (0 == 0 gives symbolic junk)
        return (RR(G(r, s, t, v, lo)), lo), (RR(G(r, s, t, v, hi)), hi)
    Gx = G(r, s, t, v)
    xs = [lo, hi]
    dGx = diff(Gx, x)
    if dGx != 0:
        for sol in solve(dGx == 0, x):
            try:
                x0 = RR(sol.right())
            except TypeError:
                continue
            if lo <= x0 <= hi:
                xs.append(x0)
    valued = [(RR(Gx(x=x0)), RR(x0)) for x0 in xs]
    return min(valued, key=lambda item: item[0]), max(valued, key=lambda item: item[0])

def min_max_H(s_1, s_2, r_2, r_1, c_minus_range=(0.2, 0.9), c_plus_range=(0.2, 0.9)):
    eta_1, eta_2 = cf_value([(0,), (1, 3)]), cf_value([(0,), (3, 1)])
    g1 = _g_extrema_on_interval(r_2, r_1, eta_1, eta_2, c_plus_range)
    g2 = _g_extrema_on_interval(eta_1, eta_2, s_1, s_2, c_minus_range)
    candidates = [
        (H(s_1, s_2, r_2, r_1, c_plus, c_minus), c_plus, c_minus)
        for _, c_plus in g1
        for _, c_minus in g2
    ]
    return min(candidates, key=lambda item: item[0]), max(candidates, key=lambda item: item[0])

def _cf_candidates(side, family):
    extensions = ((1, 3), (3, 1)) if family == "eta" else ((1, 2), (2, 1))
    return [(cf_value([(0, *side), ext]), ext) for ext in extensions]

def _pick_cf(side, family, which):
    pairs = _cf_candidates(side, family)
    return (min if which == "min" else max)(pairs, key=lambda item: item[0])

def _t_interval_expression(negative_side, neg_ext, positive_side, pos_ext):
    return (0, negative_side, neg_ext, positive_side, pos_ext)

def T_interval (negative_side, positive_side):
    # return beginning/end of T_interval I(negative_side | positive_side),
    # together with expressions witnessing each endpoint

    beginning_candidates = []
    end_candidates = []

    for pos_family, neg_family in [("eta", "theta"), ("theta", "eta")]:
        pos_b, pos_ext_b = _pick_cf(positive_side, pos_family, "min")
        neg_b, neg_ext_b = _pick_cf(negative_side, neg_family, "min")
        beginning_candidates.append((
            pos_b + neg_b,
            _t_interval_expression(negative_side, neg_ext_b, positive_side, pos_ext_b),
        ))

        pos_e, pos_ext_e = _pick_cf(positive_side, pos_family, "max")
        neg_e, neg_ext_e = _pick_cf(negative_side, neg_family, "max")
        end_candidates.append((
            pos_e + neg_e,
            _t_interval_expression(negative_side, neg_ext_e, positive_side, pos_ext_e),
        ))

    beginning, beginning_expr = min(beginning_candidates, key=lambda item: item[0])
    end, end_expr = max(end_candidates, key=lambda item: item[0])
    return beginning, beginning_expr, end, end_expr

In [ ]:
# 2.1: 1 = (-1)^{h+k} 1 < v(A) < 1.98 (i)
'''
successors = [
    ([2], [2]),
    ([3], [1, 1]),
    ([2,3], [2,3]),
    ([1, 1, 3], [3, 1]),
    ([1], []),
   
]
'''
successors = [
    ([2], [2]),
    ([3], [1, 2]),
    ([2], [1]),
    ([1], []),
   
]

# h:even k:even
h = 0
k = 0

etas = [(3,1), (1,3)]
thetas = [(2,1), (1,2)]

# initial point of the T-interval : [0;a_1, ... , a_k + r_1^j] + [0;a_{-1}, ... , a_{-h} + s_1^j] 
# end point of the T-interval : [0;a_1, ... , a_k + r_2^j] + [0;a_{-1}, ... , a_{-h} + s_2^j] 

sr_1 = [] # list of candidates of (s_1^j, r_1^j)
sr_2 = [] # list of candidates of (s_2^j, r_2^j)

for successor in successors:
    sr_1.append((
                ([(0, *successor[0]), etas[len(successor[0]) % 2]], [(0, *successor[1]), thetas[len(successor[1]) % 2]]),
                ([(0, *successor[0]), thetas[len(successor[0]) % 2]], [(0, *successor[1]), etas[len(successor[1]) % 2]]),
                ))
    sr_2.append((
                ([(0, *successor[0]), etas[(len(successor[0]) + 1) % 2]], [(0, *successor[1]), thetas[(len(successor[1]) + 1) % 2]]),
                ([(0, *successor[0]), thetas[(len(successor[0]) + 1) % 2]], [(0, *successor[1]), etas[(len(successor[1]) + 1) % 2]]),
                ))

# check (i)
# this condition seems trivial. in this case, \tilde{s}_1^j ,\tilde{r}_1^j are something like [0;(3,1)] and  [0;(2,1)]

# check (ii)
ii = check_ii(h, k, sr_1[0], sr_2[-1])
print(f"check (ii) s: {ii['s']}, r: {ii['r']}, all: {all(ii.values())}")

# calculate range of v(A) satisfying (iii)
min_vA = 1
max_vA = 1.98
for j in range(1,len(successors)):
    for s_1, r_1 in sr_1[j]:
        for s_2, r_2 in sr_2[j - 1]:
            (min_H, _, _), (max_H, _, _) = min_max_H(cf_value(s_1), cf_value(s_2), cf_value(r_2), cf_value(r_1))
            print(min_H, max_H)
            print(r_1,s_1,r_2,s_2)
            if cf_value(r_2) - cf_value(r_1) > 0:
                # TODO : h + k : odd
                if max_H > min_vA:
                    min_vA = max_H
            elif cf_value(r_2) - cf_value(r_1) < 0:
                # TODO : h + k : odd
                if min_H < max_vA:
                    max_vA = min_H

print(min_vA, max_vA)
#1 < v(A) < 0.661439607531388 ???

check (ii) s: True, r: True, all: True
0.685610630228776 0.827581983997654
[(0, 1, 2), (2, 1)] [(0, 3), (1, 3)] [(0, 2), (2, 1)] [(0, 2), (3, 1)]
0.661439607531388 0.812674301035745
[(0, 1, 2), (2, 1)] [(0, 3), (1, 3)] [(0, 2), (3, 1)] [(0, 2), (2, 1)]
0.690803207777097 0.829063570744997
[(0, 1, 2), (3, 1)] [(0, 3), (1, 2)] [(0, 2), (2, 1)] [(0, 2), (3, 1)]
0.666193061382159 0.813816397297596
[(0, 1, 2), (3, 1)] [(0, 3), (1, 2)] [(0, 2), (3, 1)] [(0, 2), (2, 1)]
0.376945368001700 0.500913297199244
[(0, 1), (1, 2)] [(0, 2), (1, 3)] [(0, 1, 2), (1, 2)] [(0, 3), (3, 1)]
0.433963582007682 0.580369430753241
[(0, 1), (1, 2)] [(0, 2), (1, 3)] [(0, 1, 2), (1, 3)] [(0, 3), (2, 1)]
0.383912726247091 0.504204041821332
[(0, 1), (1, 3)] [(0, 2), (1, 2)] [(0, 1, 2), (1, 2)] [(0, 3), (3, 1)]
0.434481166901684 0.574264360804589
[(0, 1), (1, 3)] [(0, 2), (1, 2)] [(0, 1, 2), (1, 3)] [(0, 3), (2, 1)]
0.323710834129070 0.335192753836479
[(0,), (2, 1)] [(0, 1), (1, 3)] [(0, 1), (2, 1)] [(0, 2), (3, 1)]
0.3

In [45]:
T_interval([1],[])

T_interval([1,1,3],[3,1])

(0.822835630628367,
 (0, [1, 1, 3], (1, 2), [3, 1], (3, 1)),
 0.845959400000426,
 (0, [1, 1, 3], (3, 1), [3, 1], (1, 2)))